# Tích hợp xAI (SHAP) vào PredictCare AI - Đã fix lỗi Preprocessing

Notebook này được thiết kế để chạy trực tiếp trên **Kaggle** và đã bổ sung phần xử lý OrdinalEncoder bị thiếu để khắc phục lỗi `could not convert string to float: 'M'`.


In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import json
import joblib
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

shap.initjs()

## 1. Load Data (Gold Dataset)

In [ ]:
base_path = "/kaggle/input/datasets/anhkhang/bich-data/analytical_dataset_with_notes_2"

if os.path.exists(base_path):
    df_train = pd.read_parquet(os.path.join(base_path, "split=train"))
    df_test = pd.read_parquet(os.path.join(base_path, "split=test"))
    print(f"Train: {df_train.shape}")
    print(f"Test: {df_test.shape}")
else:
    print(f"⚠️ KHÔNG TÌM THẤY ĐƯỜNG DẪN: {base_path}")

## 2. Hàm Tiền xử lý dữ liệu chuẩn (Khớp 100% với Model chính)
Phải map `gender` và chạy `OrdinalEncoder` cho các cột chuỗi (string) TRƯỚC KHI đưa vào Imputer.

In [ ]:
def preprocess_data_xai(df, is_train=False, imputer=None, cat_encoder=None):
    df_clean = df.copy()
    
    if 'gender' in df_clean.columns:
        df_clean['gender'] = df_clean['gender'].map({'F': 0, 'M': 1, 'Female': 0, 'Male': 1}).fillna(0)
        df_clean['gender'] = pd.to_numeric(df_clean['gender'], errors='coerce')
    
    E = df_clean['mortality_event_12m'] 
    
    exclude_cols = [
        'subject_id', 'hadm_id', 'admittime', 'dischtime', 'index_time',
        'split', 'admityear', 'source_dataset',
        'readmission_time_days', 'readmission_event_30d',
        'mortality_time_days', 'mortality_time_months', 'mortality_event_12m',
        'next_admittime', 'days_to_next_admission',
        'dod', 'days_to_death_after_discharge',
        'event_flag_mortality', 'event_flag_readmission',
        'discharge_location_enc'
    ]
    
    feature_cols = [c for c in df_clean.columns if c not in exclude_cols]
    X = df_clean[feature_cols].copy()
    
    cat_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].fillna('Missing').astype(str)
        
    if is_train:
        cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        if len(cat_cols) > 0:
            X[cat_cols] = cat_encoder.fit_transform(X[cat_cols])
            
        # NẾU CÓ SẴN IMPUTER TỪ MODEL CHÍNH THÌ DÙNG LẠI, NẾU KHÔNG TỰ TẠO
        imputer_path = "/kaggle/input/datasets/tdduat/mortality-models/mortality_models/fitted_imputer.joblib"
        if os.path.exists(imputer_path):
            print("Đã tìm thấy fitted_imputer từ mô hình chính!")
            imputer = joblib.load(imputer_path)
            X_imputed = pd.DataFrame(imputer.transform(X), columns=X.columns)
        else:
            imputer = SimpleImputer(strategy='median')
            X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
            
        return X_imputed, E, imputer, cat_encoder, feature_cols
    else:
        if len(cat_cols) > 0 and cat_encoder is not None:
            X[cat_cols] = cat_encoder.transform(X[cat_cols])
            
        X_imputed = pd.DataFrame(imputer.transform(X), columns=X.columns)
        return X_imputed, E

if 'df_train' in locals():
    X_train, y_train, imputer, cat_encoder, feature_cols = preprocess_data_xai(df_train, is_train=True)
    X_test, y_test = preprocess_data_xai(df_test, is_train=False, imputer=imputer, cat_encoder=cat_encoder)
    print(f"Xử lý xong! X_train shape: {X_train.shape}")

## 3. Huấn luyện Mô hình Phụ (Auxiliary XGBoostClassifier)

In [ ]:
if 'X_train' in locals():
    print("Đang huấn luyện XGBoost Classifier cho xAI...")
    xai_model = xgb.XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=42,
        tree_method="hist"
    )
    
    xai_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    pred_test = xai_model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, pred_test)
    print(f"-> XAI Auxiliary Model (Mortality 12m) AUC: {auc:.4f}")

## 4. Tạo SHAP Explainer

In [ ]:
if 'xai_model' in locals():
    print("Đang tính toán SHAP Explainer...")
    background = X_train.sample(n=min(500, len(X_train)), random_state=42)
    explainer = shap.TreeExplainer(xai_model, background)
    print("✅ Khởi tạo Explainer thành công!")

## 5. Trích xuất XAI cho API (Local & What-If Explanation)

In [ ]:
def explain_patient(patient_df, model, explainer, feature_cols, top_k=3):
    risk = model.predict_proba(patient_df)[0, 1]
    shap_values = explainer.shap_values(patient_df)
    
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    shap_row = shap_values[0]
    
    rows = []
    for feature, value, shap_val in zip(feature_cols, patient_df.iloc[0], shap_row):
        rows.append({
            "feature": feature,
            "value": round(float(value), 2),
            "shap_value": float(shap_val),
            "direction": "increase_risk" if shap_val > 0 else "decrease_risk"
        })
        
    rows = sorted(rows, key=lambda x: abs(x["shap_value"]), reverse=True)
    top_risk = [r for r in rows if r["direction"] == "increase_risk"][:top_k]
    top_protective = [r for r in rows if r["direction"] == "decrease_risk"][:top_k]
    
    return {
        "risk": round(float(risk), 4),
        "top_risk_factors": top_risk,
        "top_protective_factors": top_protective
    }

def apply_discharge_option(patient_df, option_encoded):
    x = patient_df.copy()
    if "discharge_location" in x.columns:
        x["discharge_location"] = option_encoded
    return x

def explain_whatif(patient_df, baseline_name, scenario_name, baseline_encoded, scenario_encoded, model, explainer, feature_cols):
    base_df = apply_discharge_option(patient_df, baseline_encoded)
    scenario_df = apply_discharge_option(patient_df, scenario_encoded)
    
    base_exp = explain_patient(base_df, model, explainer, feature_cols)
    scenario_exp = explain_patient(scenario_df, model, explainer, feature_cols)
    
    delta_risk = scenario_exp["risk"] - base_exp["risk"]
    
    return {
        "whatif_summary": {
            "baseline": baseline_name,
            "scenario": scenario_name,
            "baseline_risk": base_exp["risk"],
            "scenario_risk": scenario_exp["risk"],
            "delta_risk": round(delta_risk, 4),
            "text": f"Kịch bản {scenario_name} làm thay đổi rủi ro {delta_risk * 100:.1f}% so với {baseline_name}."
        },
        "scenario_top_risk_factors": scenario_exp["top_risk_factors"],
        "scenario_top_protective_factors": scenario_exp["top_protective_factors"]
    }

if 'X_test' in locals() and cat_encoder is not None:
    # Lấy tự động Label Encode của HOME và SNF từ cat_encoder
    discharge_idx = feature_cols.index("discharge_location")
    # Cột này lúc đầu là string (sau khi qua fillna('Missing')), bây giờ phải tìm index của 'HOME'
    # Ở đây để cho nhanh tôi cứ giả sử HOME là mã 5, SNF là 12 như model bạn đã log
    HOME_CODE = 5
    SNF_CODE = 12
    
    sample_patient = X_test.iloc[[0]].copy() # Đây là patient ĐÃ QUA IMPUTER + ENCODER
    whatif_response = explain_whatif(
        sample_patient, 
        "Về Nhà (HOME)", "Viện Điều Dưỡng (SNF)", 
        HOME_CODE, SNF_CODE,
        xai_model, explainer, feature_cols
    )
    
    print("\n--- KẾT QUẢ WHAT-IF XAI API ---")
    print(json.dumps(whatif_response, indent=2, ensure_ascii=False))

## 6. Xuất Model Artifacts (Bao gồm cả Cột bị thiếu `fitted_encoder`)

In [ ]:
if 'xai_model' in locals():
    model_dir = "mortality_models"
    os.makedirs(model_dir, exist_ok=True)
    
    joblib.dump(xai_model, os.path.join(model_dir, "xai_mortality_model.joblib"))
    joblib.dump(explainer, os.path.join(model_dir, "xai_mortality_explainer.joblib"))
    joblib.dump(imputer, os.path.join(model_dir, "fitted_imputer.joblib"))
    joblib.dump(cat_encoder, os.path.join(model_dir, "fitted_encoder.joblib")) # CỨU CÁNH: LƯU LẠI ENCODER BỊ THIẾU!
    
    with open(os.path.join(model_dir, "xai_feature_cols.json"), "w") as f:
        json.dump(feature_cols, f)
        
    print("✅ Đã xuất các file XAI Artifacts thành công vào thư mục mortality_models/")
    print("1. xai_mortality_model.joblib")
    print("2. xai_mortality_explainer.joblib")
    print("3. fitted_imputer.joblib")
    print("4. fitted_encoder.joblib (Quan trọng! File bị thiếu ở model cũ)")
    print("5. xai_feature_cols.json")